# 2 Customising Plots

As well as being extremely useful for creating exploratory plots, Matplotlib
can also be used to build figures for publications and reports.

## 2.1 Customising plots as you go

There are many different components of a plot that we can customise.

In this section we will cover some of the key plot elements that you may want
control over.

### Customising text

Let's start with a basic plot. We'll continue with the `electricity_emissions`
dataset from the previous chapters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jrpyvis as jrpy

In [ ]:
# Load the dataset and split into historical / Balanced Pathway
emissions = jrpy.data.load("electricity_emissions")
historical = emissions[emissions["period"] == "Historical"]
bp = emissions[emissions["period"] == "Balanced Pathway"]

In [ ]:
# Create a line plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bp["year"], bp["emissions"], c="teal")

We can customise the font and font size of any text that our figure involves.

This is done through the `fontdict` argument.

The methods that we use to control text, such as `set_title()`,
`set_[xy]label()`, `set_[xy]ticklabels()` all take this argument.

The `fontdict` parameter takes a dictionary of keyword arguments for the
configuration of a `text.Text` object.

In [ ]:
# Create a line plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bp["year"], bp["emissions"], c="teal")

# Custom title
ax.set_title(
    "UK electricity supply emissions",
    fontdict={
        "fontsize": 14,
        "fontfamily": "serif",
        "fontstyle": "italic",
        "fontweight": "bold",
    },
)

# Custom axis labels

axis_label_font = {
    "fontsize": 12,
    "fontfamily": "serif",
    "fontstyle": "italic",
}
ax.set_xlabel("Year", fontdict=axis_label_font)
ax.set_ylabel("Emissions / MtCO2e", fontdict=axis_label_font)

We can control axis tick labels with `set_[xy]ticklabels()`.

_Note_: In order to do this we first have to define the tick positions with
`set_[x,y]ticks`.

In [ ]:
# Create line plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bp["year"], bp["emissions"], c="teal")

# Set tick positions
x_tick_locs = np.arange(2025, 2055, 10)
y_tick_locs = np.arange(0, 40, 10)
ax.set_xticks(x_tick_locs)
ax.set_yticks(y_tick_locs)

# Format tick labels
ax.set_xticklabels(
    x_tick_locs,
    fontdict={"fontsize": 10, "fontfamily": "serif"},
)
ax.set_yticklabels(
    y_tick_locs,
    fontdict={"fontsize": 10, "fontfamily": "serif"},
)

It is also possible to control the appearance of the axis ticks themselves. More detail on how to do this can be found in [chapter 2 of the course notes](../notes.pdf).

## Exercises 1

Please complete Q1 of the [exercises contained here](2-exercises.ipynb#Q1a\).

## 2.2 Runtime configuration parameters

So far we have looked at how we can customise certain components of a
particular figure.

However, we can also set formatting options globally.

This is useful if we have a script which creates several figures and we want
these figures to have consistent formatting.

The default Matplotlib styling configuration is set with `matplotlib.rcParams`.

This is a dictionary containing formatting settings and their values.

By changing these values we can change default settings throughout an entire
script or notebook.

Let's check the parameters available and their current settings.

In [ ]:
import matplotlib as mpl

mpl.rcParams

We can change these settings. For example, to change default font family:

In [ ]:
mpl.rcParams["font.family"] = "serif"

We can also set multiple parameters that belong to the same family at once.

In [ ]:
mpl.rc("font", family="serif", size=12, style="italic")

If you are making a plot for a publication which contains equations, you may
wish to enable LaTeX:

In [ ]:
mpl.rcParams["text.usetex"] = True

Matplotlib will now use LaTeX to render text as long as you have LaTeX
installed.

Note that you can use a _subset_ of TeX markup in Matplotlib text strings
already.

Enabling LaTeX is more flexible though, and will allow you to use different
LaTeX packages.

To revert back to standard Matplotlib settings:

In [ ]:
mpl.rcdefaults()

## Exercises 2

Please complete Q2 of the [exercises contained here](2-exercises.ipynb#Q2a\))

## 2.3 Style sheets

Setting `rcParams` by hand every time is tedious. A **style sheet** bundles a
whole set of formatting parameters together so we can apply a consistent look in
one line.

### Inbuilt style sheets

Matplotlib has a selection of inbuilt style sheets.

In [ ]:
plt.style.available

In [ ]:
plt.style.use("ggplot")

In [ ]:
fig, ax = plt.subplots()
ax.plot(bp["year"], bp["emissions"])

A style sheet is just a text file of `rcParams`. We can see what one looks like:

In [ ]:
import os

style_dir = os.path.join(mpl.__path__[0], "mpl-data", "stylelib")
with open(os.path.join(style_dir, "ggplot.mplstyle"), "r") as f:
    print(f.read())

You can write your own `.mplstyle` file in exactly this format and load it with
`plt.style.use("<path-to-file>.mplstyle")`.

### A ready-made theme: the CCC style

Rather than build a house style from scratch, the course package ships one for
us: **`jrpyvis.ccc_theme`**. This is the CCC's own chart
style, expressed as a small Python module that sets matplotlib's `rcParams`
(their purple palette, y-axis-only gridlines, no box around the plot, and so on).

Importing it applies the style:

In [ ]:
import jrpyvis.ccc_theme as ccc

Just as we peeked inside `ggplot.mplstyle`, we can look at what this theme does.
It's a Python module rather than an `.mplstyle` file, so we read its source with
`inspect.getsource()` - showing the palette it defines and the `rcParams` it
sets:

In [ ]:
import inspect

print(inspect.getsource(ccc))

The module also gives us the CCC's named colours in `ccc.SCENARIO_COLORS` and a
helper, `ccc.zero_line()`, that draws a baseline at zero. Let's rebuild the CCC's
own electricity-supply emissions chart - historical values in dark aubergine and
the Balanced Pathway in vibrant purple:

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    historical["year"], historical["emissions"],
    color=ccc.SCENARIO_COLORS["Historical"], label="Historical",
)
ax.plot(
    bp["year"], bp["emissions"],
    color=ccc.SCENARIO_COLORS["Pathway"], label="Balanced Pathway",
)

ccc.zero_line(ax)

ax.set_xlabel("Year")
ax.set_ylabel("Emissions / MtCO2e")
ax.set_title("Electricity supply emissions - historical (2008-2023) and Balanced Pathway (2025-2050)")
ax.legend()

_This recreates the CCC's **Figure 7.5.1**; the original chart and its data are
on sheet `7.5.1` of the workbook._

## Exercises 3

Please complete Q3 of the [exercises contained here](2-exercises.ipynb#Q3a\))

[Next Chapter >>>](../chapter3/3-demo.ipynb)
